In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib.patches import Rectangle
import matplotlib.patheffects as path_effects
import matplotlib.transforms as mtransforms
from matplotlib.tri import Triangulation

import os
from copy import copy

import analysis_tools as tool
from config import *

# Load Data

In [ ]:
grid_data = tool.load_grid_data()

In [ ]:
dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

exp_name = "REA"
da_rea_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_rea_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_CTL"
da_ctl_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_ctl_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_WLT"
da_wlt_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_wlt_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_SAT"
da_sat_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_sat_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

# Precipitation Maps

In [ ]:
time_slice = slice(np.datetime64("2021-07-13T00"), np.datetime64("2021-07-15T00"))
timeframe = int((time_slice.stop - time_slice.start) / np.timedelta64(1, 'h'))

### Deterministic

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_26"], da_rea_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM, cmap="Blues", extend="max")
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_26_red"], da_sat_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM, cmap="Blues", extend="max")
ax.set(title="WET")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_26_red"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_26_red"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_26_red"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_26_red"]["clat"]) <= PLOT_WINDOW[3]))

diff = da_sat_tp.sel(step=time_slice).sum(dim="step") - da_ctl_tp.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r", extend="both")
ax.set(title="WET - CTL")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_26_red"], da_ctl_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM, cmap="Blues", extend="max")
ax.set(title="CTL")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_26_red"], da_wlt_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM, cmap="Blues", extend="max")
ax.set(title="DRY")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = da_wlt_tp.sel(step=time_slice).sum(dim="step") - da_ctl_tp.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r", extend="both")
ax.set(title="DRY - CTL")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.07, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color="white", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground="black")])
    
plt.savefig(f'./figs/figure_03.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_28"], ds_rea_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_28"], ds_sat_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Wet Conditions")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_28"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_28"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_28"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_28"]["clat"]) <= PLOT_WINDOW[3]))

diff = ds_sat_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Wet - Control")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_28"], ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Control")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_28"], ds_wlt_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Dry Conditions")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = ds_wlt_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Dry - Control")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["a", "b", "c", "d", "e", "f"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.05, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.savefig(f'./figs/map_comp_ens.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

# Time Series

## Precipitation

### Deterministic

In [ ]:
# Compute time series of deterministic runs: 
cell_areas_focus = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
#ts_rea = (da_rea_tp * grid_data["area_26"]).sel(cell=grid_data["focus_cells_26"]).sum(dim="cell")
#ts_ctl = (da_ctl_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
#ts_wlt = (da_wlt_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
#ts_sat = (da_sat_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")

cell_areas_focus_rea = grid_data["area_26"].sel(cell=grid_data["focus_cells_26"])
ts_rea = da_rea_tp.sel(cell=grid_data["focus_cells_26"]).weighted(cell_areas_focus_rea).mean(dim="cell")
ts_ctl = da_ctl_tp.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus).mean(dim="cell")
ts_wlt = da_wlt_tp.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus).mean(dim="cell")
ts_sat = da_sat_tp.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus).mean(dim="cell")

# RADOLAN observations, remapped onto grid_26 cell centers (see interpolate_radolan.py).
# Cells without radar coverage are NaN and are dropped (skipna) by the weighted mean,
# i.e. the average is over whatever fraction of the focus region RADOLAN observed.
focus_ids_26 = np.flatnonzero(grid_data["focus_cells_26"])
da_rad_tp = xr.open_dataset("data/radolan/radolan_rw_tp_grid26.nc")["tp"].reindex(cell=focus_ids_26)
ts_rad = da_rad_tp.weighted(cell_areas_focus_rea).mean(dim="cell")

In [ ]:
rea_allsum = ts_rea.sum().item()
rad_allsum = ts_rad.sum().item()

print(f"DREAM produces {(rea_allsum - rad_allsum) / rad_allsum * 100:.02f}% more/less precipitation than RADOLAN.")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_rea["valid_time"], ts_rea, color="black", label="REA")
ax.plot(ts_ctl["valid_time"], ts_ctl, color="tab:blue", label="CTL")
ax.plot(ts_wlt["valid_time"], ts_wlt, color="tab:green", label="WLT")
ax.plot(ts_sat["valid_time"], ts_sat, color="tab:orange", label="SAT")
ax.plot(ts_rad["time"], ts_rad, color="black", linestyle="--", label="RADOLAN")

plt.legend()

ax.set(ylabel="Area precipitation in mm/h")

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_deterministic.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
# Compute time series of ensemble runs:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
#ts_rea_ens = (ds_rea_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"]).sum(dim="cell")
#ts_ctl_ens = (ds_ctl_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
#ts_wlt_ens = (ds_wlt_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
#ts_sat_ens = (ds_sat_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")


cell_areas_focus_rea_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
ts_rea_ens = ds_rea_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_rea_ens).mean(dim="cell")
ts_ctl_ens = ds_ctl_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")
ts_wlt_ens = ds_wlt_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")
ts_sat_ens = ds_sat_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_ctl_ens["valid_time"], ts_ctl_ens.median(dim="mem"), color=c_ctl, label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ts_wlt_ens["valid_time"], ts_wlt_ens.median(dim="mem"), color=c_dry, label="Dry")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ts_sat_ens["valid_time"], ts_sat_ens.median(dim="mem"), color=c_wet, label="Wet")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

ax.plot(ts_rea_ens["valid_time"], ts_rea_ens.median(dim="mem"), color=c_rea, label="REA")

#plt.legend()

ax.set(ylabel="Area precipitation in kg/h")
ax.text(0.03, 0.96, "a", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_ensemble.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

In [ ]:
ctl_allsum = ts_ctl_ens.median(dim="mem").sum().item()
rea_allsum = ts_rea_ens.median(dim="mem").sum().item()
wlt_allsum = ts_wlt_ens.median(dim="mem").sum().item()
sat_allsum = ts_sat_ens.median(dim="mem").sum().item()

print(f"CTL (median) produces {(ctl_allsum - rea_allsum) / rea_allsum * 100:.02f}% more/less precipitation than DREAM (median).")
print(f"SAT (median) produces {(sat_allsum - ctl_allsum) / ctl_allsum * 100:.02f}% more/less precipitation than CTL (median).")
print(f"WLT (median) produces {(wlt_allsum - ctl_allsum) / ctl_allsum * 100:.02f}% more/less precipitation than CTL (median).")

## Soil Moisture

In [ ]:
dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

# Native grid_28 extpar (same grid as the ensemble W_SO): soil type + land fraction
extpar_28 = xr.open_dataset("invar/icon_extpar_0028_R02B07_N02_20150521.nc")

# Geographic (Mid-Europe) selection ...
region = ((np.rad2deg(grid_data["grid_28"]["clon"]) >= PRUDENCE_REGIONS["ME"]["lon"].start) &
          (np.rad2deg(grid_data["grid_28"]["clon"]) <= PRUDENCE_REGIONS["ME"]["lon"].stop) &
          (np.rad2deg(grid_data["grid_28"]["clat"]) >= PRUDENCE_REGIONS["ME"]["lat"].start) &
          (np.rad2deg(grid_data["grid_28"]["clat"]) <= PRUDENCE_REGIONS["ME"]["lat"].stop))

is_land = xr.DataArray(extpar_28["FR_LAND"].values > 0.5, dims=region.dims)
mask = region & is_land

In [ ]:
def compute_wso_iqr(exp_name, base_dir, dts, mask):
    """Just a helper function to limit memory usage."""
    ds_ens_wso = tool.read_merged_var_ens("W_SO", dts, f"{base_dir}/{exp_name}/merged/W_SO", accu=False)
    _ = ds_ens_wso.isel(cell=mask, depthBelowLandLayer=0).mean(dim="cell")
    
    pct_25 = _.quantile(0.25, dim="mem")
    pct_50 = _.quantile(0.5, dim="mem")
    pct_75 = _.quantile(0.75, dim="mem")

    return pct_25, pct_50, pct_75

rea_25, rea_50, rea_75 = compute_wso_iqr("REA", base_dir, dts, mask)
ctl_25, ctl_50, ctl_75 = compute_wso_iqr("BLK_CTL", base_dir, dts, mask)
wlt_25, wlt_50, wlt_75 = compute_wso_iqr("BLK_WLT", base_dir, dts, mask)
sat_25, sat_50, sat_75 = compute_wso_iqr("BLK_SAT", base_dir, dts, mask)

In [ ]:
# --- Reference soil-moisture levels from TERRA soil hydraulic constants ---
# By soil type index (1..10): ice, rock, sand, sandy loam, loam, loam-clay, clay, peat, sea water, sea ice
cporv = np.array([1e-10, 1e-10, 0.364, 0.445, 0.455, 0.475, 0.507, 0.863, 1e-10, 1e-10])  # pore volume (saturation)
cfcap = np.array([1e-10, 1e-10, 0.196, 0.260, 0.340, 0.370, 0.463, 0.763, 1e-10, 1e-10])  # field capacity
cpwp  = np.array([0.0,   0.0,   0.042, 0.100, 0.110, 0.185, 0.257, 0.265, 0.0,   0.0])     # permanent wilting point

# Soil-layer thicknesses (m) matching the W_SO depthBelowLandLayer boundaries
# [0, 0.01, 0.03, 0.09, 0.27, 0.81, 2.43, 7.29, 21.87]
dz_soil = np.array([0.01, 0.02, 0.06, 0.18, 0.54, 1.62, 4.86, 14.58])
RHO_W = 1000.0  # density of water [kg/m^3]

# Native soil type on the ensemble grid_28 (exact, same cell order as the W_SO output -> no interpolation)
soiltyp_28 = extpar_28["SOILTYP"].values.astype(int)
is_soil_28 = (soiltyp_28 >= 3) & (soiltyp_28 <= 8)
fc_vol_28  = np.where(is_soil_28, cfcap[soiltyp_28 - 1], np.nan)
pwp_vol_28 = np.where(is_soil_28, cpwp[soiltyp_28 - 1],  np.nan)
sat_vol_28 = np.where(is_soil_28, cporv[soiltyp_28 - 1], np.nan)

# Convert the *top* layer (matching depthBelowLandLayer=0 used above) to W_SO units [kg/m^2]
# and average over the same cells as the time series (land soil cells within `mask`).
_sel = mask.values & is_soil_28
sat_top = np.nanmean(sat_vol_28[_sel]) * dz_soil[0] * RHO_W
fc_top  = np.nanmean(fc_vol_28[_sel])  * dz_soil[0] * RHO_W
pwp_top = np.nanmean(pwp_vol_28[_sel]) * dz_soil[0] * RHO_W
print(f"Regional top-layer reference [kg/m^2]:  saturation={sat_top:.3f}  "
      f"field capacity={fc_top:.3f}  wilting point={pwp_top:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.plot(ctl_50["valid_time"], ctl_50, color="tab:blue", label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color="tab:blue", alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color="tab:green", label="Dry")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color="tab:green", alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color="tab:orange", label="Wet")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color="tab:orange", alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color="black", label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color="black", alpha=0.3)

# Reference levels from soil hydraulic properties (regional mean over soil cells)
ax.axhline(sat_top, color="dimgray", ls=":",  lw=1.0, label="Saturation")
ax.axhline(fc_top,  color="dimgray", ls="--", lw=1.3, label="Field capacity")
ax.axhline(pwp_top, color="dimgray", ls="-.", lw=1.0, label="Wilting point")

plt.legend()

ax.set(ylabel=r"Average soil moisture in kg/m$^2$")
ax.text(0.03, 0.96, "b", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold",
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_soil_moisture.png', dpi=300, bbox_inches='tight', format='png')
plt.show()


In [ ]:
ctl_smm = ctl_50.median().item()
rea_smm = rea_50.median().item()
wlt_smm = wlt_50.median().item()
sat_smm = sat_50.median().item()

print(f"CTL (median) has {(ctl_smm - rea_smm) / rea_smm * 100:.02f}% more/less top level soil moisture than DREAM (median).")
print(f"SAT (median) has {(sat_smm - ctl_smm) / ctl_smm * 100:.02f}% more/less top level soil moisture than CTL (median).")
print(f"WLT (median) has {(wlt_smm - ctl_smm) / ctl_smm * 100:.02f}% more/less top level soil moisture than CTL (median).")

### As Index

In [ ]:
def compute_wso_iqr(exp_name, base_dir, dts, mask):
    """Just a helper function to limit memory usage."""
    ds_ens_wso = tool.read_merged_var_ens("W_SO", dts, f"{base_dir}/{exp_name}/merged/W_SO", accu=False)

    # Fraction of each soil layer lying within the top 1 m (same convention as the
    # smi_top1m map below): 1.0 for fully-included layers, 0 for fully-excluded ones,
    # and the partial fraction for the one layer straddling the 1 m boundary.
    depth_bnds = np.concatenate([[0.0], np.cumsum(dz_soil)])
    depth_frac = np.clip((1.0 - depth_bnds[:-1]) / dz_soil, 0.0, 1.0)
    frac_weights = xr.DataArray(depth_frac, coords={"depthBelowLandLayer": ds_ens_wso["depthBelowLandLayer"].values})

    # Depth-integrate to a top-1 m volumetric soil moisture [m^3/m^3]: W_SO per layer is a mass
    # total for that layer (kg/m^2), so the fractionally-included layers must be summed and
    # normalized by the *target* depth (1 m), not averaged across layers.
    theta_1m = (ds_ens_wso.isel(cell=mask) * frac_weights).sum(dim="depthBelowLandLayer") / (RHO_W * 1.0)

    smi = (theta_1m - pwp_vol_28[mask])/(fc_vol_28 - pwp_vol_28)[mask]
    smi = smi.mean(dim="cell")

    pct_25 = smi.quantile(0.25, dim="mem")
    pct_50 = smi.quantile(0.5, dim="mem")
    pct_75 = smi.quantile(0.75, dim="mem")

    return pct_25, pct_50, pct_75

rea_25, rea_50, rea_75 = compute_wso_iqr("REA", base_dir, dts, mask & is_soil_28)
ctl_25, ctl_50, ctl_75 = compute_wso_iqr("BLK_CTL", base_dir, dts, mask & is_soil_28)
wlt_25, wlt_50, wlt_75 = compute_wso_iqr("BLK_WLT", base_dir, dts, mask & is_soil_28)
sat_25, sat_50, sat_75 = compute_wso_iqr("BLK_SAT", base_dir, dts, mask & is_soil_28)

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.plot(ctl_50["valid_time"], ctl_50, color="tab:blue", label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color="tab:blue", alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color="tab:green", label="Dry")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color="tab:green", alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color="tab:orange", label="Wet")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color="tab:orange", alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color="black", label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color="black", alpha=0.3)

plt.legend()

ax.set(ylabel=r"Soil Moisture Index")
ax.text(0.03, 0.96, "b", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold",
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_soil_moisture.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

## Precip & SM in one plot

In [ ]:
fig, axs = plt.subplots(2,1, figsize=(8,6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
fig.tight_layout(h_pad=0.8)

# Precip
ax = axs[0]
ax.plot(ts_ctl_ens["valid_time"], ts_ctl_ens.median(dim="mem"), color="tab:blue", label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color="tab:blue", alpha=0.3)

ax.plot(ts_wlt_ens["valid_time"], ts_wlt_ens.median(dim="mem"), color="tab:green", label="Dry")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color="tab:green", alpha=0.3)

ax.plot(ts_sat_ens["valid_time"], ts_sat_ens.median(dim="mem"), color="tab:orange", label="Wet")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color="tab:orange", alpha=0.3)

ax.plot(ts_rea_ens["valid_time"], ts_rea_ens.median(dim="mem"), color="black", label="REA")

ax.legend()

ax.set(ylabel=r"Area precipitation in kg h$^{-1}$")
ax.text(0.02, 0.96, "(a)", transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# SM
ax = axs[1]
ax.plot(ctl_50["valid_time"], ctl_50, color="tab:blue", label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color="tab:blue", alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color="tab:green", label="Dry")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color="tab:green", alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color="tab:orange", label="Wet")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color="tab:orange", alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color="black", label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color="black", alpha=0.3)

#plt.legend()

ax.set(ylabel=r"Average soil moisture in kg m$^{-2}$")
ax.text(0.02, 0.9, "(b)", transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


plt.xticks(rotation=30)
plt.savefig(f'./figs/figure_04.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

## Evaporation

In [ ]:
R = 6371000  # Earth radius (m)
lat_spacing = 0.15
lon_spacing = 0.15

# Convert to radians
dlat = np.deg2rad(lat_spacing)
dlon = np.deg2rad(lon_spacing)

# Compute area
ll_grid = xr.open_dataset("data/moisture_tracking/REA/expanded_region.nc").rename({"latitude": "lat", "longitude": "lon"})

lat_rad = np.deg2rad(ll_grid["lat"])
area = R**2 * dlon * np.abs(np.sin(lat_rad + dlat/2) - np.sin(lat_rad - dlat/2))
area_mt = area.reindex({"lat": area["lat"][::-1]}) + 0 * ll_grid["lon"] #hack to broadcast to lat/lon

fr_land_mt = xr.open_dataset("data/moisture_tracking/invar/fr_land_015x015_EUL.nc")["FR_LAND"]
fr_land_mt = fr_land_mt.reindex({"lat": fr_land_mt["lat"][::-1]}).squeeze()

In [ ]:
def read_mt_output(exp_name, var):
    """
    Helper function to declutter the renaming and reindexing operations for reading tracking output data.
    """
    da_track = xr.open_mfdataset(f"data/moisture_tracking/{exp_name}/output/det/backtrack_2021-07-??T00-00.nc")[var]
    da_track = da_track.rename({"latitude": "lat", "longitude": "lon"})
    da_track = da_track.reindex({"lat": da_track["lat"][::-1]})
    return da_track

In [ ]:
da_rea_track = read_mt_output("REA", "e_track")
da_ctl_track = read_mt_output("BLK_CTL", "e_track")
da_sat_track = read_mt_output("BLK_SAT", "e_track")
da_wlt_track = read_mt_output("BLK_WLT", "e_track")

In [ ]:
mask_evapt = fr_land_mt.where(fr_land_mt > 0.8)
contribution = (da_rea_track + da_ctl_track + da_wlt_track + da_sat_track).sum(dim="time")
shares_track = contribution / contribution.max()  # ~1 for the most-contributing cells, not normalized to sum to 1
weights_evapt = area_mt.values * shares_track #minor coordinate misalignment in area compared to shares

In [ ]:
da_rea_evapt = xr.open_mfdataset("data/moisture_tracking/REA/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_rea_evapt = -da_rea_evapt.reindex({"lat": da_rea_evapt["lat"][::-1]}) * 3600  # kg m^-2 s^-1 -> mm h^-1 (1 kg/m^2 == 1 mm)

da_ctl_evapt = xr.open_mfdataset("data/moisture_tracking/BLK_CTL/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_ctl_evapt = -da_ctl_evapt.reindex({"lat": da_ctl_evapt["lat"][::-1]}) * 3600

da_sat_evapt = xr.open_mfdataset("data/moisture_tracking/BLK_SAT/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_sat_evapt = -da_sat_evapt.reindex({"lat": da_sat_evapt["lat"][::-1]}) * 3600

da_wlt_evapt = xr.open_mfdataset("data/moisture_tracking/BLK_WLT/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_wlt_evapt = -da_wlt_evapt.reindex({"lat": da_wlt_evapt["lat"][::-1]}) * 3600

# weights_evapt's lat/lon come from expanded_region.nc and differ from the EVAPT files' own
# lat/lon at float precision -- force exact match so .weighted() doesn't silently drop cells
# via xarray's exact-label alignment (all 4 experiments share identical EVAPT lat/lon).
weights_evapt = weights_evapt.assign_coords(lat=da_ctl_evapt["lat"], lon=da_ctl_evapt["lon"])

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.plot(da_ctl_evapt["time"], da_ctl_evapt.weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_ctl, label="CTL")

ax.plot(da_wlt_evapt["time"], da_wlt_evapt.weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_dry, label="Dry")

ax.plot(da_sat_evapt["time"], da_sat_evapt.weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_wet, label="Wet")

ax.plot(da_rea_evapt["time"], da_rea_evapt.weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_rea, label="REA")

plt.legend()

ax.set(ylabel=r"$\langle$ EVAPT $\rangle_\text{w}$ in mm h$^{-1}$")

plt.xticks(rotation=30)
plt.savefig(f'./figs/figure_06.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

## CAPE

In [ ]:
ds_cape_ctl = xr.open_mfdataset("data/BLK_CTL/merged/CAPE/fc_R03B07_cape_merged.202107????")["cape"]
ds_cape_ctl_ens = xr.concat([xr.open_mfdataset(f"data/BLK_CTL/merged/CAPE/fc_R02B07_N02_cape_merged.202107????.{mmm:03}") for mmm in range(1,21)], dim="mem")["cape"]

ds_cape_wlt = xr.open_mfdataset("data/BLK_WLT/merged/CAPE/fc_R03B07_cape_merged.202107????")["cape"]
ds_cape_wlt_ens = xr.concat([xr.open_mfdataset(f"data/BLK_WLT/merged/CAPE/fc_R02B07_N02_cape_merged.202107????.{mmm:03}") for mmm in range(1,21)], dim="mem")["cape"]

ds_cape_sat = xr.open_mfdataset("data/BLK_SAT/merged/CAPE/fc_R03B07_cape_merged.202107????")["cape"]
ds_cape_sat_ens = xr.concat([xr.open_mfdataset(f"data/BLK_SAT/merged/CAPE/fc_R02B07_N02_cape_merged.202107????.{mmm:03}") for mmm in range(1,21)], dim="mem")["cape"]

In [ ]:
lons, lats = np.rad2deg(grid_data["grid_26_red"]["clon"]), np.rad2deg(grid_data["grid_26_red"]["clat"])
mask = (lons > PASSIVE_REGION["x0"]) & (lons < PASSIVE_REGION["x1"]) & (lats > PASSIVE_REGION["y0"]) & (lats < PASSIVE_REGION["y1"])

tri_passive = Triangulation(lons[mask].values, lats[mask].values)

mask_cape_26 = ((lons[mask] > FOCUS_REGION["x0"]) 
              & (lons[mask] < FOCUS_REGION["x1"]) 
              & (lats[mask] > FOCUS_REGION["y0"])
              & (lats[mask] < FOCUS_REGION["y1"]))

lons, lats = np.rad2deg(grid_data["grid_28"]["clon"]), np.rad2deg(grid_data["grid_28"]["clat"])
mask = (lons > PASSIVE_REGION["x0"]) & (lons < PASSIVE_REGION["x1"]) & (lats > PASSIVE_REGION["y0"]) & (lats < PASSIVE_REGION["y1"])
mask_cape_28 = ((lons[mask] > FOCUS_REGION["x0"]) 
              & (lons[mask] < FOCUS_REGION["x1"]) 
              & (lats[mask] > FOCUS_REGION["y0"])
              & (lats[mask] < FOCUS_REGION["y1"]))

In [ ]:
fig, ax = plt.subplots()

ax.plot(ds_cape_ctl["time"], ds_cape_ctl.sel(cell=mask_cape_26).mean(dim="cell"), label="CTL")
ax.plot(ds_cape_wlt["time"], ds_cape_wlt.sel(cell=mask_cape_26).mean(dim="cell"), label="WLT")
ax.plot(ds_cape_sat["time"], ds_cape_sat.sel(cell=mask_cape_26).mean(dim="cell"), label="SAT")

plt.legend()
plt.xticks(rotation=45)
ax.set(ylabel="CAPE in J/kg")

plt.show()

## TP Det+IQR & CP Det+IQR & CAPE & SM & Evap

In [ ]:
cell_areas_focus_rea_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
ts_rea_ens = ds_rea_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_rea_ens).mean(dim="cell")
ts_ctl_ens = ds_ctl_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")
ts_wlt_ens = ds_wlt_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")
ts_sat_ens = ds_sat_ens_tp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")

In [ ]:
def get_cp_ts(dts, base_dir):
    """Helper function to limit memory usage."""
    exp_name = "BLK_CTL"
    da_ctl_cp = tool.read_merged_var_det("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)
    ds_ctl_ens_cp = tool.read_merged_var_ens("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)

    exp_name = "BLK_WLT"
    da_wlt_cp = tool.read_merged_var_det("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)
    ds_wlt_ens_cp = tool.read_merged_var_ens("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)

    exp_name = "BLK_SAT"
    da_sat_cp = tool.read_merged_var_det("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)
    ds_sat_ens_cp = tool.read_merged_var_ens("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)

    # Compute time series of deterministic runs: 
    cell_areas_focus = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
    ts_ctl_cp = da_ctl_cp.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus).mean(dim="cell")
    ts_wlt_cp = da_wlt_cp.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus).mean(dim="cell")
    ts_sat_cp = da_sat_cp.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus).mean(dim="cell")

    # Compute time series of ensemble runs:
    cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
    ts_ctl_ens_cp = ds_ctl_ens_cp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")
    ts_wlt_ens_cp = ds_wlt_ens_cp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")
    ts_sat_ens_cp = ds_sat_ens_cp.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell")

    return( ts_ctl_cp, ts_wlt_cp, ts_sat_cp, ts_ctl_ens_cp, ts_wlt_ens_cp, ts_sat_ens_cp )

dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")
base_dir = "/automount/agh/s6tifohr/july21_eval/data"
ts_ctl_cp, ts_wlt_cp, ts_sat_cp, ts_ctl_ens_cp, ts_wlt_ens_cp, ts_sat_ens_cp = get_cp_ts(dts, base_dir)

In [ ]:
fig, axs = plt.subplots(5,1, figsize=(8,10), sharex=True, gridspec_kw={'height_ratios': [2, 2*4/9, 1, 1, 1]})
fig.tight_layout(h_pad=0.8)
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans)

# TP
ax = axs[0]
ax.plot(ts_ctl["valid_time"], ts_ctl, color=c_ctl, label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ts_wlt["valid_time"], ts_wlt, color=c_dry, label="DRY")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ts_sat["valid_time"], ts_sat, color=c_wet, label="WET")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

ax.plot(ts_rea["valid_time"], ts_rea, color="black", label="REA")

ax.legend()
#ax.set_ylim(0, 7e11)
ax.set(ylabel=r"$\langle$ TP $\rangle$ in mm h$^{-1}$")
ax.text(0, 1, "(a)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# CP
ax = axs[1]
ax.plot(ts_ctl_cp["valid_time"], ts_ctl_cp, color=c_ctl, label="CTL")
ax.fill_between(ts_ctl_ens_cp["valid_time"], ts_ctl_ens_cp.quantile(0.25, dim="mem"), ts_ctl_ens_cp.quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ts_wlt_cp["valid_time"], ts_wlt_cp, color=c_dry, label="DRY")
ax.fill_between(ts_wlt_ens_cp["valid_time"], ts_wlt_ens_cp.quantile(0.25, dim="mem"), ts_wlt_ens_cp.quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ts_sat_cp["valid_time"], ts_sat_cp, color=c_wet, label="WET")
ax.fill_between(ts_sat_ens_cp["valid_time"], ts_sat_ens_cp.quantile(0.25, dim="mem"), ts_sat_ens_cp.quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

#ax.set_ylim(0, 3e11)
ax.set(ylabel=r"$\langle$ CP $\rangle$ in mm h$^{-1}$")
ax.text(0, 1, "(b)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# CAPE
ax = axs[2]
ax.plot(ds_cape_ctl["time"], ds_cape_ctl.sel(cell=mask_cape_26).mean(dim="cell"), color=c_ctl, label="CTL")
ax.fill_between(ds_cape_ctl_ens["time"], 
                ds_cape_ctl_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.25, dim="mem"), 
                ds_cape_ctl_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ds_cape_wlt["time"], ds_cape_wlt.sel(cell=mask_cape_26).mean(dim="cell"), color=c_dry, label="DRY")
ax.fill_between(ds_cape_wlt_ens["time"], 
                ds_cape_wlt_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.25, dim="mem"), 
                ds_cape_wlt_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ds_cape_sat["time"], ds_cape_sat.sel(cell=mask_cape_26).mean(dim="cell"), color=c_wet, label="WET")
ax.fill_between(ds_cape_sat_ens["time"], 
                ds_cape_sat_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.25, dim="mem"), 
                ds_cape_sat_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

ax.set(ylabel=r"$\langle$ CAPE $\rangle$ in J kg$^{-1}$")
ax.text(0, 1, "(c)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# SM
ax = axs[3]
ax.plot(ctl_50["valid_time"], ctl_50, color=c_ctl, label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color=c_ctl, alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color=c_dry, label="DRY")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color=c_dry, alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color=c_wet, label="WET")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color=c_wet, alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color=c_rea, label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color=c_rea, alpha=0.3)

#ax.set(ylabel=r"$\langle$ SM $\rangle$ in kg m$^{-2}$")
ax.set(ylabel="Soil Moisture Index")
ax.text(0, 1, "(d)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# EVAPT
ax = axs[4]
ax.plot(da_ctl_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_ctl_evapt.sel(time=slice(dts[0], dts[-1])).weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_ctl, label="CTL")
ax.plot(da_wlt_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_wlt_evapt.sel(time=slice(dts[0], dts[-1])).weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_dry, label="Dry")
ax.plot(da_sat_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_sat_evapt.sel(time=slice(dts[0], dts[-1])).weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_wet, label="Wet")
ax.plot(da_rea_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_rea_evapt.sel(time=slice(dts[0], dts[-1])).weighted(weights_evapt).mean(dim=["lat", "lon"]), color=c_rea, label="REA")

ax.set(ylabel=r"$\langle$ EVAPT $\rangle_\text{w}$ in mm h$^{-1}$")
ax.text(0, 1, "(e)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


plt.xticks(rotation=30)
plt.savefig(f'./figs/figure_04.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Scatter Plot

In [ ]:
dts = pd.date_range("2021-07-13T00", "2021-07-15T00", freq="3h")

# PRMSL
#ds_prmsl_rea = tool.read_merged_var_det("prmsl", dts, "data/REA/merged/prmsl", accu=False) / 100.

ds_prmsl_ctl = tool.read_merged_var_det("prmsl", dts, "data/BLK_CTL/merged/prmsl", accu=False) / 100.
ds_prmsl_ctl_ens = tool.read_merged_var_ens("prmsl", dts, "data/BLK_CTL/merged/prmsl", accu=False) / 100.

ds_prmsl_sat = tool.read_merged_var_det("prmsl", dts, "data/BLK_SAT/merged/prmsl", accu=False) / 100.
ds_prmsl_sat_ens = tool.read_merged_var_ens("prmsl", dts, "data/BLK_SAT/merged/prmsl", accu=False) / 100.

ds_prmsl_wlt = tool.read_merged_var_det("prmsl", dts, "data/BLK_WLT/merged/prmsl", accu=False) / 100.
ds_prmsl_wlt_ens = tool.read_merged_var_ens("prmsl", dts, "data/BLK_WLT/merged/prmsl", accu=False) / 100.


# TOT_PRECIP
#ds_tp_rea = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/REA/merged/tp", accu=True)

ds_tp_ctl = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_CTL/merged/tp", accu=True)
ds_tp_ctl_ens = tool.read_merged_var_ens("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_CTL/merged/tp", accu=True)

ds_tp_wlt = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_WLT/merged/tp", accu=True)
ds_tp_wlt_ens = tool.read_merged_var_ens("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_WLT/merged/tp", accu=True)

ds_tp_sat = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_SAT/merged/tp", accu=True)
ds_tp_sat_ens = tool.read_merged_var_ens("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_SAT/merged/tp", accu=True)


# GPH
ds_gph_ctl = tool.read_merged_var_det("gph", dts, "data/BLK_CTL/merged/gph", accu=False, format="netcdf").sel(plev=50000)
ds_gph_ctl_ens = tool.read_merged_var_ens("gph", dts, "data/BLK_CTL/merged/gph", accu=False, format="netcdf").sel(plev=50000)

ds_gph_sat = tool.read_merged_var_det("gph", dts, "data/BLK_SAT/merged/gph", accu=False, format="netcdf").sel(plev=50000)
ds_gph_sat_ens = tool.read_merged_var_ens("gph", dts, "data/BLK_SAT/merged/gph", accu=False, format="netcdf").sel(plev=50000)

ds_gph_wlt = tool.read_merged_var_det("gph", dts, "data/BLK_WLT/merged/gph", accu=False, format="netcdf").sel(plev=50000)
ds_gph_wlt_ens = tool.read_merged_var_ens("gph", dts, "data/BLK_WLT/merged/gph", accu=False, format="netcdf").sel(plev=50000)

In [ ]:
# Compute precip sums DET:
cell_areas_focus_det = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
lats, lons = np.rad2deg(grid_data["grid_26_red"]["clat"].values), np.rad2deg(grid_data["grid_26_red"]["clon"].values)
mask_mean_26_red = (lons > FOCUS_REGION["x0"]-2) & (lons < FOCUS_REGION["x1"]+2) & (lats > FOCUS_REGION["y0"]-2) & (lats < FOCUS_REGION["y1"]+2)

#sums_rea_det = (ds_tp_rea.sel(cell=grid_data["focus_cells_26"]) * cell_areas_focus_det).sum(dim=["cell", "step"])
sums_ctl_det = ds_tp_ctl.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus_det).mean(dim="cell").sum(dim="step")
sums_sat_det = ds_tp_sat.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus_det).mean(dim="cell").sum(dim="step")
sums_wlt_det = ds_tp_wlt.sel(cell=grid_data["focus_cells_26_red"]).weighted(cell_areas_focus_det).mean(dim="cell").sum(dim="step")


# Compute precip sums ENS:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
lats, lons = np.rad2deg(grid_data["grid_28"]["clat"].values), np.rad2deg(grid_data["grid_28"]["clon"].values)
mask_mean_28 = (lons > FOCUS_REGION["x0"]-2) & (lons < FOCUS_REGION["x1"]+2) & (lats > FOCUS_REGION["y0"]-2) & (lats < FOCUS_REGION["y1"]+2)

sums_ctl_ens = ds_tp_ctl_ens.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell").sum(dim="step")
sums_sat_ens = ds_tp_sat_ens.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell").sum(dim="step")
sums_wlt_ens = ds_tp_wlt_ens.sel(cell=grid_data["focus_cells_28"]).weighted(cell_areas_focus_ens).mean(dim="cell").sum(dim="step")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9,4), sharey=True)
fig.tight_layout(h_pad=2)
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans) #to position text consistently


# PSML
ax = axs[0]
ax.set_xlim(983, 1007)

# CONTROL
x, y = ds_prmsl_ctl_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_ctl_ens
ax.scatter(x, y, color=c_ctl, s=25, label="CTL")
ax.scatter(ds_prmsl_ctl.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_ctl_det, color=c_ctl, edgecolors="white", s=70, marker="X")

# WET
x, y = ds_prmsl_sat_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_sat_ens
ax.scatter(x, y, color=c_wet, s=25, label="WET")
ax.scatter(ds_prmsl_sat.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_sat_det, color=c_wet, edgecolors="white", s=70, marker="X")

# DRY
x, y = ds_prmsl_wlt_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_wlt_ens
ax.scatter(x, y, color=c_dry, s=25, label="DRY")
ax.scatter(ds_prmsl_wlt.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_wlt_det, color=c_dry, edgecolors="white", s=70, marker="X")

# labels
ax.set(xlabel=r"Min[p$_\text{msl}$]$_\text{A,t}$ in hPa", ylabel=r"$\sum$ TP in mm")
ax.text(0, 1, "(a)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# GPH
ax = axs[1]
ax.set_xlim(5460, 5690)

# CONTROL
x, y = ds_gph_ctl_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_ctl_ens
ax.scatter(x, y, color=c_ctl, label="CTL")
ax.scatter(ds_gph_ctl.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_ctl_det, color=c_ctl, edgecolors="white", s=70, marker="X", zorder=10)

# WET
x, y = ds_gph_sat_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_sat_ens
ax.scatter(x, y, color=c_wet, label="WET")
ax.scatter(ds_gph_sat.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_sat_det, color=c_wet, edgecolors="white", s=70, marker="X", zorder=10)

# DRY
x, y = ds_gph_wlt_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_wlt_ens
ax.scatter(x, y, color=c_dry, label="DRY")
ax.scatter(ds_gph_wlt.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_wlt_det, color=c_dry, edgecolors="white", s=70, marker="X", zorder=10)

# labels
ax.set(xlabel=r"Min[h$_\text{500}$]$_\text{A,t}$ in m")
ax.text(0, 1, "(b)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])
ax.legend()


plt.savefig(f'./figs/figure_05.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Member Differences in PMSL and W

## PSML

In [ ]:
DT = "2021-07-14T18"
PDIFF_LVL = np.arange(-6, 7, 1)
PDIFF_NORM = colors.TwoSlopeNorm(vmin=-6, vcenter=0, vmax=6)

diffs = {"WET - CTL": (ds_prmsl_sat_ens - ds_prmsl_ctl_ens).sel(step=DT),
         "DRY - CTL": (ds_prmsl_wlt_ens - ds_prmsl_ctl_ens).sel(step=DT)}
ctl_mean = ds_prmsl_ctl_ens.sel(step=DT).mean("mem")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 5), subplot_kw={"projection": ccrs.PlateCarree()})
fig.tight_layout()
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans) #to position text consistently

for ax, (title, d) in zip(axs, diffs.items()):
    dmean = d.mean("mem").values
    im = ax.tricontourf(grid_data["tri_28"], dmean, levels=PDIFF_LVL,
                        norm=PDIFF_NORM, cmap="bwr")

    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.set(title=title)

axs[0].text(0, 1, "(a)", transform=axs[0].transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])

axs[1].text(0, 1, "(b)", transform=axs[1].transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


fig.colorbar(im, ax=axs, fraction=0.019, pad=0.04, label="Ensemble-mean MSLP difference in hPa")
plt.savefig(f'./figs/figure_06.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

## Vertical Velocity

In [ ]:
# Vertical velocity is chosen at two discrete levels:
# half level 59 -> ca. 2km height (lower troposphere)
# half level 35 -> ca. 9km height (upper troposphere)
LVL_L, LVL_U = 59, 35

def read_wz_levels(data_dir, dts, levels=(LVL_L, LVL_U)):
    """Like tool.read_merged_var_ens, but selects only `levels` of wz before
    loading."""
    members = []
    for mem in range(1, 21):
        steps = []
        for dt in dts:
            f = (f"{data_dir}/fc_R02B07_N02_wz_merged."
                 f"{dt.year}{dt.month:02}{dt.day:02}{dt.hour:02}.{mem:03}")
            da = xr.open_dataset(f, engine="cfgrib",
                                 backend_kwargs={"indexpath": ""})["wz"]
            da = da.sel(generalVertical=list(levels))     # <- keep only 2 levels
            if da.sizes["step"] == 4:                     # drop analysis step
                da = da.isel(step=slice(1, None))
            da["step"] = da["valid_time"]
            steps.append(da.rename({"values": "cell"}).load())
        members.append(xr.concat(steps, dim="step"))
    return xr.concat(members, dim="mem")

dts_w = pd.date_range("2021-07-14T12", "2021-07-14T18", freq="3h")

wz_ctl = read_wz_levels("data/BLK_CTL/merged/wz", dts_w)
wz_sat = read_wz_levels("data/BLK_SAT/merged/wz", dts_w)
wz_wlt = read_wz_levels("data/BLK_WLT/merged/wz", dts_w)

w_low_ctl, w_upp_ctl = wz_ctl.sel(generalVertical=LVL_L), wz_ctl.sel(generalVertical=LVL_U)
w_low_sat, w_upp_sat = wz_sat.sel(generalVertical=LVL_L), wz_sat.sel(generalVertical=LVL_U)
w_low_wlt, w_upp_wlt = wz_wlt.sel(generalVertical=LVL_L), wz_wlt.sel(generalVertical=LVL_U)

In [ ]:
from scipy.interpolate import griddata

# Points to interpolate to:
gx = np.arange(-5, 5.01, 0.2)
gy = np.arange(-5, 5.01, 0.2)
GX, GY = np.meshgrid(gx, gy)

# Region in which to search for / evaluate the surface low:
#reg_low  = (lons > 0) & (lons < 20) & (lats > 44) & (lats < 58)
reg_low  = (lons > FOCUS_REGION["x0"]-2) & (lons < FOCUS_REGION["x1"]+2) & (lats > FOCUS_REGION["y0"]-2) & (lats < FOCUS_REGION["y1"]+2)

def composite_on_low(field, locator):
    """Center each member on `locator`'s (MSLP) minimum, composite `field`.
    Because of the unstructured grid, interpolation is required to compare
    members and their different low centres. 
    Note: `acc` and `cnt` are functionally just computing the mean in case
    their might be missing data."""
    acc = np.zeros_like(GX); cnt = np.zeros_like(GX)
    for i in range(1,20):
        p_i = locator.isel(mem=i).sel(step=DT).values
        j = np.nanargmin(np.where(reg_low, p_i, np.nan))
        lon0, lat0 = lons[j], lats[j]
        f_i = field.isel(mem=i).sel(step=DT).values
        near = (np.abs(lons - lon0) < 10) & (np.abs(lats - lat0) < 10)
        vals = griddata((lons[near], lats[near]), f_i[near],
                        (lon0 + GX, lat0 + GY), method="linear")
        m = np.isfinite(vals); acc[m] += vals[m]; cnt[m] += 1
    return acc / np.where(cnt > 0, cnt, np.nan)

# Each experiment is centered on its own low, then differenced,
# so displacement of the low is not mistaken for a change in ascent strength.
rows = [("Lower trop.", w_low_ctl, w_low_sat, w_low_wlt),
        ("Upper trop.", w_upp_ctl, w_upp_sat, w_upp_wlt)]

fig, axs = plt.subplots(2, 2, figsize=(9,8), sharex=True, sharey=True)
fig.tight_layout()
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans) #to position text consistently

for r, (rlab, wc, ws, ww) in enumerate(rows):

    # Convert velocity to centimeters per second
    cc    = 100 * composite_on_low(wc, ds_prmsl_ctl_ens)
    d_sat = 100 * composite_on_low(ws, ds_prmsl_sat_ens) - cc
    d_wlt = 100 * composite_on_low(ww, ds_prmsl_wlt_ens) - cc

    lv = np.linspace(-12, 12, 25)
    
    for ax, comp, clab in zip(axs[r], [d_sat, d_wlt], ["WET − CTL", "DRY − CTL"]):
        im = ax.contourf(gx, gy, comp, levels=lv, cmap="RdBu_r", extend="both")
        ax.contour(gx, gy, comp, levels=[0], colors="k", linewidths=0.6)
        ax.set_aspect("equal")
        ax.axhline(0, color="k", alpha=0.5)
        ax.axvline(0, color="k", alpha=0.5)

        if r == 0:
            ax.set_title(clab)

fig.colorbar(im, ax=axs, orientation="vertical", fraction=0.025, pad=0.02,
                label=r"$\Delta$w in cm s$^{-1}$")
for ax in axs[-1]:
    ax.set_xlabel(r"Distance from low centre in °E")
for ax in axs[:, 0]:
    ax.set_ylabel(r"Distance from low centre in °N")
for ax, label in zip(axs.reshape(-1), ["(a)", "(b)", "(c)", "(d)"]):
    ax.text(0, 1, label, transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])

# Add row labels on the left
fig.text(-0.02, 0.75, "Lower troposphere", transform=fig.transFigure,
         ha="right", va="center", fontsize=11, rotation=90, fontweight="bold")
fig.text(-0.02, 0.25, "Upper troposphere", transform=fig.transFigure,
         ha="right", va="center", fontsize=11, rotation=90, fontweight="bold")

plt.savefig(f'./figs/figure_07.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Assimilation Region Showcase

In [ ]:
# This is not actually coinciding with the passive region, which is -10 to 40E and 30 to 75N, 
# but I used it to get a background in the image below  
mask_passive_region = ((np.rad2deg(grid_data["grid_26"]["clon"]) >= -20) & (np.rad2deg(grid_data["grid_26"]["clon"]) <= 50) & 
                       (np.rad2deg(grid_data["grid_26"]["clat"]) >= 20) & (np.rad2deg(grid_data["grid_26"]["clat"]) <= 85)).values
lons_pr, lats_pr = np.rad2deg(grid_data["grid_26"]["clon"][mask_passive_region]), np.rad2deg(grid_data["grid_26"]["clat"][mask_passive_region])

ii_passive_region = np.arange(len(mask_passive_region))[mask_passive_region]

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([-30, 70, 25, 80], crs=ccrs.PlateCarree())

gl1 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.15, zorder=10, color="black")
gl1.top_labels = False
gl1.right_labels = False

im = ax.tricontourf(grid_data["tri_28"], np.zeros(len(grid_data["area_28"])))
ax.text(50, 32, "EU Nest", color="white", weight="bold")

# Create a rectangle patch for the passive observation region:
box_passive = Rectangle((PASSIVE_REGION["x0"], PASSIVE_REGION["y0"]), PASSIVE_REGION["wx"], PASSIVE_REGION["wy"], edgecolor="purple", linewidth=2, fill=False)
ax.add_patch(box_passive)
ax.text(-9, 72, "Passive Observations", color="purple", weight="bold")

# Create a rectangle patch for the focus region
box_focus = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor="red", linewidth=2, fill=False)
ax.add_patch(box_focus)
ax.text(11, 50, "Focus Region", color="red", weight="bold")

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)

plt.savefig("figs/figure_01.png", dpi=300, bbox_inches='tight', format='png')
plt.show()

# Predictability

Get cell areas on lat-lon grid:

In [ ]:
R = 6371000  # Earth radius (m)
lat_spacing = 0.2
lon_spacing = 0.2

# Convert to radians
dlat = np.deg2rad(lat_spacing)
dlon = np.deg2rad(lon_spacing)

# Compute area
ll_grid = xr.open_dataset("./data/free_forecasts/2021071200/mem001/fc_ll_DOM02_0001.nc")

lat_rad = np.deg2rad(ll_grid["lat"])
area = R**2 * dlon * np.abs(np.sin(lat_rad + dlat/2) - np.sin(lat_rad - dlat/2))
area_fc = area + 0 * ll_grid["lon"] #hack to broadcast to lat/lon

Need mid-points of the focus region:

In [ ]:
mx = (FOCUS_REGION["x0"] + FOCUS_REGION["x1"])/2
my = (FOCUS_REGION["y0"] + FOCUS_REGION["y1"])/2
mt = np.datetime64("2021-07-14T00")

Precipitation sums in free forecasts

In [ ]:
init_sums = {} #sums for all initialization dates
dts_fc = pd.date_range("2021-07-01T00", "2021-07-12T00", freq="D")

for dt in dts_fc:
    print(dt)
    
    ens_sums = np.full(20, np.nan) #maximum sum for all members of one initialization date

    for mem in range(1,21):
        path = f"./data/free_forecasts/{dt.year}{dt.month:02}{dt.day:02}00/mem{mem:03}/"
        fnames = [path + fname for fname in sorted(os.listdir(path))[-97:]]
        da_mem = xr.open_mfdataset(fnames)["tot_prec"].diff(dim="time")        

        area_sum = area_fc.rolling(lon=int(FOCUS_REGION["wx"]/lon_spacing), lat=int(FOCUS_REGION["wy"]/lat_spacing), center=True).sum()

        da_mem = da_mem.sel(lon=slice(FOCUS_REGION["x0"] - 3, FOCUS_REGION["x1"] + 3), 
                            lat=slice(FOCUS_REGION["y0"] - 3, FOCUS_REGION["y1"] + 3), 
                            time=slice(np.datetime64("2021-07-12"), np.datetime64("2021-07-16")), 
                            drop=True)

        rolling_sum = ((da_mem * area_fc).rolling(lon=int(FOCUS_REGION["wx"]/lon_spacing), lat=int(FOCUS_REGION["wy"]/lat_spacing), time=48, center=True).sum() / area_sum).compute()

        cutout = rolling_sum.sel(lon=slice(mx-1, mx+1), lat=slice(my-1, my+1), time=slice(mt-np.timedelta64(12,"h"), mt+np.timedelta64(12,"h")))

        # If the indeces are needed:
        #ii_max = cutout.argmax(..., skipna=True)
        #cutout_max = cutout.isel(time=ii_max["time"], lat=ii_max["lat"], lon=ii_max["lon"])
        #x0_max, y0_max = cutout_max["lon"] - wx/2, cutout_max["lat"] - wy/2

        ens_sums[mem-1] = cutout.max(skipna=True)

    init_sums[dt] = ens_sums

Precipitation sums in storyline scenarios:

In [ ]:
time_slice = slice(np.datetime64("2021-07-13T00"), np.datetime64("2021-07-15T00"))

#rea_sums = (ds_rea_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
#ctl_sums = (ds_ctl_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
#sat_sums = (ds_sat_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
#wlt_sums = (ds_wlt_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])


cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
rea_sums = ds_rea_ens_tp.sel(cell=grid_data["focus_cells_28"], step=time_slice).weighted(cell_areas_focus_ens).mean("cell").sum("step")
ctl_sums = ds_ctl_ens_tp.sel(cell=grid_data["focus_cells_28"], step=time_slice).weighted(cell_areas_focus_ens).mean("cell").sum("step")

Amount of moisture backtracked:

In [ ]:
fnames_rea = [f"data/moisture_tracking/REA/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_rea = xr.open_mfdataset(fnames_rea)
ds_track_rea = ds_track_rea.reindex(latitude=list(reversed(ds_track_rea["latitude"])))

area_weights = np.cos(np.deg2rad(ds_track_rea["latitude"]))

sum_tagged = ds_track_rea["tagged_precip"].weighted(area_weights).sum().values
ts_evap = ds_track_rea["e_track"].weighted(area_weights).sum(dim=["latitude", "longitude"])
ts_evap = ts_evap[::-1].cumsum()[::-1]
ts_loss = ds_track_rea["losses"].weighted(area_weights).sum(dim=["latitude", "longitude"])

In [ ]:
fig, ax1 = plt.subplots()


# Plot share of evaporation:
ax1.plot(range(1,13), ts_evap[:12] / sum_tagged)
ax1.set_ylim(0,1)
ax1.set_yticks(np.arange(0, 1.1, 0.1))
ax1.set(ylabel="Share of precipitation tracked to source")

# x orientation:
ax1.tick_params(axis="x", labelrotation=45)


# Add explaination to FC init.
x1, x2 = 1, 12
y1, y2 = -0.15, -0.18

ax1.vlines([x1, x2], y1-0.02, y1+0.02, colors='k', lw=1.5,
          transform=ax1.get_xaxis_transform(), clip_on=False)
ax1.plot([x1, x2], [y1, y1], 'k-', lw=1.5, clip_on=False,
        transform=ax1.get_xaxis_transform())

ax1.text((x1+x2)/2, y2, "Forecast initialization date", ha="center", va="top",
        transform=ax1.get_xaxis_transform())


# Plot simulated precipitation sums:
ax2 = ax1.twinx()

x = [init_sums[dt] for dt in init_sums.keys()] + [rea_sums.values, ctl_sums.values]#, wlt_sums.values, sat_sums.values]
tick_labels = [f"{dt.day}.{dt.month}." for dt in dts_fc] + ["DREAM", "LDA CTL"]#, "LDA Dry", "LDA Wet"] #tick labels are overriden by boxplot

bp = ax2.boxplot(x, tick_labels=tick_labels)
ax2.set(ylabel=r"$\langle$ TP $\rangle$ in mm")
#ax2.set_ylim(0, 2.1e13)

# Highlight ICON-DREAM:
idx, lw = -2, 1.5
bp["boxes"][idx].set_linewidth(lw)
bp["whiskers"][2*idx].set_linewidth(lw)
bp["whiskers"][2*idx+1].set_linewidth(lw)
bp["caps"][2*idx].set_linewidth(lw)
bp["caps"][2*idx+1].set_linewidth(lw)


plt.savefig(f'./figs/figure_02.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Moisture Tracking

In [ ]:
fnames_rea = [f"data/moisture_tracking/REA/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_rea = xr.open_mfdataset(fnames_rea)
ds_track_rea = ds_track_rea.reindex(latitude=list(reversed(ds_track_rea["latitude"])))

fnames_ctl = [f"data/moisture_tracking/BLK_CTL/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_ctl = xr.open_mfdataset(fnames_ctl)
ds_track_ctl = ds_track_ctl.reindex(latitude=list(reversed(ds_track_ctl["latitude"])))

fnames_wlt = [f"data/moisture_tracking/BLK_WLT/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_wlt = xr.open_mfdataset(fnames_wlt)
ds_track_wlt = ds_track_wlt.reindex(latitude=list(reversed(ds_track_wlt["latitude"])))

fnames_sat = [f"data/moisture_tracking/BLK_SAT/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_sat = xr.open_mfdataset(fnames_sat)
ds_track_sat = ds_track_sat.reindex(latitude=list(reversed(ds_track_sat["latitude"])))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10,5), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=1, h_pad=1)

ax = axs[0,0]
im = ax.contourf(ds_track_rea["longitude"], ds_track_rea["latitude"], ds_track_rea["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="ICON-DREAM")

ax = axs[0,1]
im = ax.contourf(ds_track_sat["longitude"], ds_track_sat["latitude"], ds_track_sat["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="WET")

ax = axs[1,0]
im = ax.contourf(ds_track_ctl["longitude"], ds_track_ctl["latitude"], ds_track_ctl["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="CTL")

ax = axs[1,1]
im = ax.contourf(ds_track_wlt["longitude"], ds_track_wlt["latitude"], ds_track_wlt["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="DRY")


for ax, char in zip(axs.reshape(-1), ["(a)", "(b)", "(c)", "(d)"]):
    ax.add_feature(cfeature.COASTLINE)
    ax.set_extent([-80, 30, 25, 65], crs=ccrs.PlateCarree())    #Bounds: West, East, South, North

    ax.text(0.05, 0.86, char, transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground="white")])
    
    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

plt.subplots_adjust(bottom=0.08)
cbar_ax = fig.add_axes([0.35, 0.02, 0.6, 0.03])
cbar = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")

fig.text(0.15, 0.02, "Tagged Evaporation in mm")

plt.savefig(f'./figs/figure_06.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

In [ ]:
fr_land = xr.open_dataset("data/moisture_tracking/invar/fr_land_015x015_EUL.nc")["FR_LAND"]
fr_land = fr_land.squeeze().drop_vars("time")
fr_land = fr_land.reindex(lat=list(reversed(fr_land["lat"]))).rename({"lat": "latitude", "lon": "longitude"})
fr_land = fr_land.assign_coords(latitude=fr_land["latitude"].round(6), longitude=fr_land["longitude"].round(6))

sea_regions = {}

# Mediterranean Sea:
mask_1 = ((fr_land["longitude"] >= -6) & (fr_land["longitude"] <= 37) & (fr_land["latitude"] >= 30) & (fr_land["latitude"] <= 41))
mask_2 = ((fr_land["longitude"] >= 0) & (fr_land["longitude"] <= 27) & (fr_land["latitude"] >= 40) & (fr_land["latitude"] <= 48))
sea_regions["MeS"] = {"name": "Mediterranean Sea", "mask": copy((mask_1 | mask_2) & (fr_land < 0.1))}

# Black Sea:
#mask = ((fr_land["longitude"] >= 27) & (fr_land["longitude"] <= 43) & (fr_land["latitude"] >= 40) & (fr_land["latitude"] <= 48))
#sea_regions["BlS"] = {"name": "Black Sea", "mask": copy(mask & (fr_land < 0.1))}

# Caspian Sea:
#mask = ((fr_land["longitude"] >= 46) & (fr_land["longitude"] <= 56) & (fr_land["latitude"] >= 36) & (fr_land["latitude"] <= 48))
#sea_regions["CaS"] = {"name": "Caspian Sea", "mask": copy(mask & (fr_land < 0.1))}

# Baltic Sea:
mask_1 = ((fr_land["longitude"] >= 10) & (fr_land["longitude"] <= 32) & (fr_land["latitude"] >= 52) & (fr_land["latitude"] <= 60))
mask_2 = ((fr_land["longitude"] >= 12) & (fr_land["longitude"] <= 29) & (fr_land["latitude"] >= 52) & (fr_land["latitude"] <= 70))
mask = mask_1 | mask_2
sea_regions["BaS"] = {"name": "Baltic Sea", "mask": copy((mask_1 | mask_2) & (fr_land < 0.1))}

# North Sea:
mask_1 = ((fr_land["longitude"] >= -3) & (fr_land["longitude"] <= 9) & (fr_land["latitude"] >= 53) & (fr_land["latitude"] <= 70))
mask_2 = ((fr_land["longitude"] >= -5) & (fr_land["longitude"] <= 12) & (fr_land["latitude"] >= 60) & (fr_land["latitude"] <= 70))
mask_3 = ((fr_land["longitude"] >= -5) & (fr_land["longitude"] <= 10) & (fr_land["latitude"] >= 55) & (fr_land["latitude"] <= 70))
mask = mask_1 | mask_2 | mask_3
sea_regions["NoS"] = {"name": "North Sea", "mask": copy((mask_1 | mask_2 | mask_3) & (fr_land < 0.1))}

# West European Shelf:
mask_1 = ((fr_land["longitude"] >= -3) & (fr_land["longitude"] <= 5) & (fr_land["latitude"] >= 45) & (fr_land["latitude"] <= 53))
mask_2 = ((fr_land["longitude"] >= -9) & (fr_land["longitude"] <= -1) & (fr_land["latitude"] >= 42) & (fr_land["latitude"] <= 55))
mask_3 = ((fr_land["longitude"] >= -12) & (fr_land["longitude"] <= -5) & (fr_land["latitude"] >= 50) & (fr_land["latitude"] <= 60))
mask = mask_1 | mask_2 | mask_3
sea_regions["WES"] = {"name": "West European Shelf", "mask": copy((mask_1 | mask_2 | mask_3) & (fr_land < 0.1))}

# Labrador Sea:
mask = ((fr_land["longitude"] >= -65) & (fr_land["longitude"] <= -42) & (fr_land["latitude"] >= 52) & (fr_land["latitude"] <= 65))
sea_regions["LaS"] = {"name": "Caspian Sea", "mask": copy(mask & (fr_land < 0.1))}

# North Atlantic:
mask_hudson = ((fr_land["longitude"] >= -80) & (fr_land["longitude"] <= -60) & (fr_land["latitude"] >= 50) & (fr_land["latitude"] <= 65))
mask = ((fr_land["longitude"] <= 0) & ~sea_regions["WES"]["mask"] & ~sea_regions["NoS"]["mask"] & 
        ~sea_regions["MeS"]["mask"] & ~sea_regions["LaS"]["mask"] & ~mask_hudson)
sea_regions["NAt"] = {"name": "North Atlantic", "mask": copy(mask & (fr_land < 0.1))}

del sea_regions["LaS"]

In [ ]:
area_weights = np.cos(np.deg2rad(ds_track_rea["latitude"]))
total_tracked = (area_weights * ds_track_rea["e_track"]).sum()
total_tagged  = (area_weights * ds_track_rea["tagged_precip"]).sum()
total_tracked / total_tagged * 100

In [ ]:
contributions = {}

for exp, dataset in zip(["CTL", "REA", "WLT", "SAT"], [ds_track_ctl, ds_track_rea, ds_track_wlt, ds_track_sat]):
    total = dataset["e_track"].weighted(area_weights).sum().values.item()
    unaccounted = (dataset["tagged_precip"].weighted(area_weights).sum().values.item() - total) #/ dataset["tagged_precip"].sum().values.item()

    _ = {}#{"unacc": unaccounted}

    for key in PRUDENCE_REGIONS:
        evap = dataset["e_track"].sel(latitude=PRUDENCE_REGIONS[key]["lat"], longitude=PRUDENCE_REGIONS[key]["lon"])
        evap_land = evap.where(fr_land.sel(latitude=PRUDENCE_REGIONS[key]["lat"], longitude=PRUDENCE_REGIONS[key]["lon"]).values > 0.5, other=0.)
        _[key] = evap_land.weighted(area_weights).sum().values.item() / total

    for key in sea_regions:
        _[key] = dataset["e_track"].where(sea_regions[key]["mask"], other=0.).weighted(area_weights).sum().values.item() / total

    contributions[exp] = _

In [ ]:
keys = list(contributions[list(contributions.keys())[0]].keys())
x = np.arange(len(keys))
width = 0.2

for i, (name, data) in enumerate(contributions.items()):
    values = [data[k] for k in keys]
    plt.bar(x + i*width, values, width, label=name)

plt.xticks(x + width*1.5, keys)
plt.legend()
plt.show()

In [ ]:
for key in ["REA", "CTL", "WLT", "SAT"]:  
    line = f"{key}"

    for val in contributions[key].values():
        line += f" & {100 * val:.01f}"
    
    print(line + " \\\\")

# TQV Map

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import AxesGrid
from cartopy.mpl.geoaxes import GeoAxes
import os
import contextlib

path = "/automount/agh/s6tifohr/july21_eval/data/"

with open(os.devnull, 'w') as fnull:
    with contextlib.redirect_stderr(fnull):
        CTL = xr.open_dataset(path + "BLK_CTL/2021071412/fc_R03B07_rea_ml.2021071412", engine="cfgrib", backend_kwargs=({"indexpath":""}))
        SAT = xr.open_dataset(path + "BLK_SAT/2021071412/fc_R03B07_rea_ml.2021071412", engine="cfgrib", backend_kwargs=({"indexpath":""}))
        WLT = xr.open_dataset(path + "BLK_WLT/2021071412/fc_R03B07_rea_ml.2021071412", engine="cfgrib", backend_kwargs=({"indexpath":""}))    

In [ ]:
colors = ['#eff3ff', '#bdd7e7', '#6baed6', '#3182bd', '#08519c', '#08306b', '#08306b', 'yellow', 'gold', 'orange', 'darkorange', 'coral']
cmap = LinearSegmentedColormap.from_list("tqv_cmap", list(zip(np.linspace(0, 1, len(colors)), colors)))
levels = np.arange(0, 60, 5)

fig = plt.figure(figsize=(10, 6))
axes_class = (GeoAxes, {"projection": ccrs.PlateCarree()})
axgr = AxesGrid(fig, 111, axes_class=axes_class, nrows_ncols=(1, 3),
                axes_pad=0.3, cbar_location="right", cbar_mode="single",
                cbar_pad=0.1, cbar_size="4%", label_mode="L")

for ax, data, title, char in zip(axgr, [CTL["TQV"], SAT["TQV"], WLT["TQV"]], ["CTL", "WET", "DRY"], ["(a)", "(b)", "(c)"]):
    cf = ax.tricontourf(grid_data["tri_26_red"], data, levels=levels, cmap=cmap, extend="neither")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="white")
    ax.set_extent([-9, 45, 36, 71], crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=14)

    ax.text(0.05, 0.94, char, transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2.5, foreground="white")])

axgr.cbar_axes[0].colorbar(cf)
axgr.cbar_axes[0].set_ylabel(r"TQV in kg m$^{-2}$", fontsize=13)

plt.savefig(f'./figs/figure_A02.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Soil Moisture Index (top 1 m)

Soil moisture index following mHm (Feddes stress factor),
$\mathrm{SMI} = \dfrac{\theta - \theta_\mathrm{pwp}}{\theta_\mathrm{fc} - \theta_\mathrm{pwp}}$,
i.e. 0 at the permanent wilting point and 1 at field capacity. Computed from the
deterministic run over the top 1 m of soil.

In [ ]:
# --- Soil moisture index (SMI) over the top 1 m of soil, deterministic run ---
# SMI = (theta - theta_pwp) / (theta_fc - theta_pwp)   (mHm / Feddes: 0 = wilting point, 1 = field capacity)
smi_exp = "REA"                          # deterministic run to show
smi_dt  = pd.Timestamp("2021-07-10T00")  # analysis time

# Native soil type on the deterministic grid_26 (exact, same cell order as the grid / W_SO output)
extpar_det = xr.open_dataset("invar/icon_extpar_0026_R03B07_G_20140731.nc")
soiltyp_det = extpar_det["SOILTYP"].values.astype(int)
is_soil_det = (soiltyp_det >= 3) & (soiltyp_det <= 8)
fc_vol_det  = np.where(is_soil_det, cfcap[soiltyp_det - 1], np.nan)
pwp_vol_det = np.where(is_soil_det, cpwp[soiltyp_det - 1],  np.nan)

# Deterministic W_SO at the analysis time (global grid_26)
wso_det = xr.open_dataset(f"{base_dir}/{smi_exp}/merged/W_SO/fc_R03B07_W_SO_merged.{smi_dt:%Y%m%d%H}",
                          engine="cfgrib", backend_kwargs={"indexpath": ""})["W_SO"].isel(step=-1).values

# Fraction of each soil layer that lies within the top 1 m, then depth-integrate to volumetric SM
_bnds = np.concatenate([[0.0], np.cumsum(dz_soil)])      # layer bottom boundaries [m]
_frac = np.clip((1.0 - _bnds[:-1]) / dz_soil, 0.0, 1.0)
theta_1m = (wso_det * _frac[:, None]).sum(axis=0) / (RHO_W * 1.0)   # top-1 m volumetric SM [m^3/m^3]

# fc/pwp are depth-independent, so their top-1 m average equals the tabulated values
smi = (theta_1m - pwp_vol_det) / (fc_vol_det - pwp_vol_det)

fig, ax = plt.subplots(figsize=(7, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent([-11, 30, 35, 68], crs=ccrs.PlateCarree())

gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.15, zorder=10, color="black")
gl.top_labels = False
gl.right_labels = False

tri_smi = copy(grid_data["tri_26"])                     # mask triangles that touch non-soil cells
tri_smi.set_mask(np.any(~np.isfinite(smi)[tri_smi.triangles], axis=1))
im = ax.tricontourf(tri_smi, np.where(np.isfinite(smi), smi, 0.0),
                    levels=np.linspace(0, 1, 11), cmap="BrBG", extend="both")

ax.add_feature(cfeature.OCEAN, zorder=2)
ax.add_feature(cfeature.COASTLINE, zorder=3)
rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"],
                      edgecolor="red", linewidth=2, fill=False, zorder=4)
ax.add_patch(rectangle)

cbar = fig.colorbar(im, ax=ax, shrink=0.8, extend="both")
cbar.set_label("Soil moisture index (top 1 m)")
ax.set(title=f"{smi_exp} soil moisture index — {smi_dt:%Y-%m-%d %H} UTC")

plt.savefig("./figs/smi_top1m.png", dpi=300, bbox_inches="tight", format="png")
plt.show()


# Requested Maps: Evaporation, E-P, Humidity & Temperature

Each shown as the mean WET/DRY - CTL difference over the two
weeks leading up to the event (2021-07-01 to 2021-07-14), on the same lat/lon grid as the
moisture tracking output above. 2 m temperature instead has to use the deterministic
output, which is only available from 2021-07-10 onward, so its averaging window is shorter.

In [ ]:
def load_daily_var(exp_name, var_file, var_name, dts):
    """Lazily loads a daily moisture-tracking-preprocessing variable (e.g. EVAPT, TOT_PREC, QV) for one experiment."""
    fnames = [f"data/moisture_tracking/{exp_name}/det/icon_R03B07_{var_file}_{dt.strftime('%Y%m%d')}.nc" for dt in dts]
    return xr.open_mfdataset(fnames)[var_name]


def nice_step(span, target_bins=8):
    """Rounds span/target_bins up to a 'nice' 1/2/2.5/5 x10^n step, so levels land on round numbers."""
    if span <= 0:
        return 1.0
    raw_step = span / target_bins
    exp = np.floor(np.log10(raw_step))
    base = raw_step / 10**exp
    for mult in (1, 2, 2.5, 5, 10):
        if base <= mult:
            return round(mult * 10**exp, 10)
    return 10**(exp + 1)


def plot_wetdry_diff_maps(diff_wet, diff_dry, cbar_label, savepath, lon=None, lat=None, tri=None, mask_window=None):
    """
    Plots WET-CTL and DRY-CTL difference maps side by side. Pass either lon/lat for a
    regular grid (contourf) or tri + mask_window for the ICON triangular grid (tricontourf).

    The color scale is calibrated from data inside `plot_window` only and rounded to a 'nice'
    step so the 0-boundary and colorbar tick labels are round numbers instead of raw floats.
    """
    if tri is not None:
        if mask_window is not None:
            diff_wet, diff_dry = diff_wet.copy(), diff_dry.copy()
            diff_wet.data[~mask_window] = 0.
            diff_dry.data[~mask_window] = 0.
        vis_wet, vis_dry = diff_wet, diff_dry
    else:
        window_mask = ((lon >= plot_window[0]) & (lon <= plot_window[1]) &
                        (lat >= plot_window[2]) & (lat <= plot_window[3]))
        vis_wet, vis_dry = diff_wet.where(window_mask), diff_dry.where(window_mask)

    vmax_raw = float(max(np.nanmax(np.abs(vis_wet)), np.nanmax(np.abs(vis_dry))))
    step = nice_step(2 * vmax_raw)
    vmax = np.ceil(vmax_raw / step) * step
    levels = np.round(np.arange(-vmax, vmax + step / 2, step), 10)
    norm = colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    fig, axs = plt.subplots(1, 2, figsize=(9, 4.2), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout(w_pad=2.5)

    for ax, diff, title, char in zip(axs, [diff_wet, diff_dry], ["WET - CTL", "DRY - CTL"], ["(a)", "(b)"]):
        if tri is not None:
            im = ax.tricontourf(tri, diff, levels=levels, norm=norm, cmap="coolwarm_r", extend="both")
        else:
            im = ax.contourf(lon, lat, diff, levels=levels, norm=norm, cmap="coolwarm_r", extend="both")
        ax.set(title=title)

        ax.set_extent(plot_window, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.OCEAN)
        ax.add_feature(cfeature.COASTLINE)

        rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
        ax.add_patch(rectangle)

        ax.text(0.07, 0.93, char, transform=ax.transAxes,
            ha="center", va="center", fontweight="bold",
            color="white", zorder=1,
            path_effects=[path_effects.withStroke(linewidth=3, foreground="black")])

    # Position must be read AFTER set_extent: cartopy's aspect-locking can grow the geo-axes box
    # beyond what tight_layout originally allotted, so measuring it beforehand places the
    # colorbar too high and it ends up overlapping the maps. The colorbar's *top* edge (its y0
    # plus its height) must sit strictly below pos.y0, not just its y0 -- otherwise the box
    # itself still pokes up into the maps regardless of how small the nominal gap looks.
    pos = axs[0].get_position()
    cbar_gap, cbar_height = 0.015, 0.03
    cbar_ax = fig.add_axes([0.3, pos.y0 - cbar_gap - cbar_height, 0.4, cbar_height])
    fig.colorbar(im, cax=cbar_ax, orientation="horizontal", label=cbar_label, ticks=levels)

    plt.savefig(savepath, dpi=300, bbox_inches='tight', format='png')
    plt.show()


# Two weeks leading up to the event, matching the moisture tracking window used above:
dts_track = pd.date_range("2021-07-01", "2021-07-14", freq="D")

# Wider regional extent used only for this reviewer-response section (rest of the notebook keeps
# using the narrower PLOT_WINDOW from config.py):
plot_window = [-11, 35, 36, 65]

## Evaporation

In [ ]:
da_evapt_ctl = -load_daily_var("BLK_CTL", "e", "EVAPT", dts_track) * 3600  # kg m^-2 s^-1 -> mm h^-1, sign flip so positive = evaporation
da_evapt_sat = -load_daily_var("BLK_SAT", "e", "EVAPT", dts_track) * 3600
da_evapt_wlt = -load_daily_var("BLK_WLT", "e", "EVAPT", dts_track) * 3600

evapt_diff_wet = (da_evapt_sat - da_evapt_ctl).mean(dim="time").compute()
evapt_diff_dry = (da_evapt_wlt - da_evapt_ctl).mean(dim="time").compute()

In [ ]:
plot_wetdry_diff_maps(
    evapt_diff_wet, evapt_diff_dry,
    cbar_label=r"$\Delta$E in mm h$^{-1}$",
    savepath="./figs/figure_s1.png",
    lon=evapt_diff_wet["lon"], lat=evapt_diff_wet["lat"],
)

## Evaporation - Precipitation

In [ ]:
da_tp_ctl = load_daily_var("BLK_CTL", "tp", "TOT_PREC", dts_track)  # already an hourly rate in mm h^-1
da_tp_sat = load_daily_var("BLK_SAT", "tp", "TOT_PREC", dts_track)
da_tp_wlt = load_daily_var("BLK_WLT", "tp", "TOT_PREC", dts_track)

ep_ctl = (da_evapt_ctl - da_tp_ctl).mean(dim="time").compute()
ep_sat = (da_evapt_sat - da_tp_sat).mean(dim="time").compute()
ep_wlt = (da_evapt_wlt - da_tp_wlt).mean(dim="time").compute()

In [ ]:
# Shown as CTL/WET/DRY absolute maps rather than WET/DRY-CTL differences: the raw balance is
# easier to read than a difference-of-a-difference, and it directly shows where each scenario
# is evaporation-dominated (E > P) versus precipitation-dominated (E < P).
window_mask = ((ep_ctl["lon"] >= plot_window[0]) & (ep_ctl["lon"] <= plot_window[1]) &
               (ep_ctl["lat"] >= plot_window[2]) & (ep_ctl["lat"] <= plot_window[3]))
vis = xr.concat([ep_ctl, ep_sat, ep_wlt], dim="exp").where(window_mask)
vmin_raw, vmax_raw = float(np.nanmin(vis)), float(np.nanmax(vis))
step = nice_step(vmax_raw - vmin_raw)
vmin = np.floor(vmin_raw / step) * step
vmax = np.ceil(vmax_raw / step) * step
levels = np.round(np.arange(vmin, vmax + step / 2, step), 10)  # 0 is always a multiple of `step`, so it is an exact level boundary
norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)  # red-to-blue transition lands exactly on the 0 level

fig, axs = plt.subplots(2, 2, figsize=(9, 7.5), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=2.5, h_pad=2.5)

# The bottom-right quadrant hosts the colorbar rather than a map: grab its footprint now, before
# removing it, so the colorbar can be sized/centered within that quadrant instead of stretching
# across it (the other axes' cartopy aspect-locking only resizes their own box, so this quadrant's
# position is unaffected regardless of when it's read).
cbar_quadrant = axs[1, 1].get_position()
axs[1, 1].remove()

map_axs = [axs[0, 0], axs[0, 1], axs[1, 0]]
for ax, field, title, char in zip(map_axs, [ep_ctl, ep_sat, ep_wlt], ["CTL", "WET", "DRY"], ["(a)", "(b)", "(c)"]):
    im = ax.contourf(ep_ctl["lon"], ep_ctl["lat"], field, levels=levels, norm=norm, cmap="coolwarm_r", extend="both")
    ax.set(title=title)

    ax.set_extent(plot_window, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.07, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold",
        color="white", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground="black")])

# Horizontal colorbar with a normal aspect ratio (short and wide), centered in its quadrant
# rather than filling it, so it reads like an ordinary standalone colorbar instead of a
# stretched bar.
cbar_width, cbar_height = cbar_quadrant.width * 0.75, 0.025
cbar_x = cbar_quadrant.x0 + (cbar_quadrant.width - cbar_width) / 2
cbar_y = cbar_quadrant.y0 + (cbar_quadrant.height - cbar_height) / 2
cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height])
fig.colorbar(im, cax=cbar_ax, orientation="horizontal", label="E - P in mm h$^{-1}$", ticks=levels)

plt.savefig("./figs/figure_s2.png", dpi=300, bbox_inches='tight', format='png')
plt.show()

## Lowest-Level Humidity

In [ ]:
# Reads the full 3-D specific humidity fields file-by-file, then keeps only the lowest model
# level (height index 120 of 120) before averaging, so it takes a couple of minutes to run.
da_qv_ctl = load_daily_var("BLK_CTL", "ml_q", "QV", dts_track).isel(height=-1) * 1000  # kg/kg -> g/kg
da_qv_sat = load_daily_var("BLK_SAT", "ml_q", "QV", dts_track).isel(height=-1) * 1000
da_qv_wlt = load_daily_var("BLK_WLT", "ml_q", "QV", dts_track).isel(height=-1) * 1000

qv_diff_wet = (da_qv_sat - da_qv_ctl).mean(dim="time").compute()
qv_diff_dry = (da_qv_wlt - da_qv_ctl).mean(dim="time").compute()

In [ ]:
plot_wetdry_diff_maps(
    qv_diff_wet, qv_diff_dry,
    cbar_label=r"$\Delta q_v$ in g kg$^{-1}$",
    savepath="./figs/figure_s3.png",
    lon=qv_diff_wet["lon"], lat=qv_diff_wet["lat"],
)

## 2 m Temperature

In [ ]:
plot_window = [-11, 35, 36, 65]

base_dir_t2m = "/automount/agh/s6tifohr/july21_eval/data"
dts_t2m = pd.date_range("2021-07-10T00", "2021-07-15T21", freq="3h")  # merged output only starts on the 10th

da_t2m_ctl = tool.read_merged_var_det("t2m", dts_t2m, f"{base_dir_t2m}/BLK_CTL/merged/t2m", accu=False)
da_t2m_sat = tool.read_merged_var_det("t2m", dts_t2m, f"{base_dir_t2m}/BLK_SAT/merged/t2m", accu=False)
da_t2m_wlt = tool.read_merged_var_det("t2m", dts_t2m, f"{base_dir_t2m}/BLK_WLT/merged/t2m", accu=False)

t2m_diff_wet = (da_t2m_sat - da_t2m_ctl).mean(dim="step")
t2m_diff_dry = (da_t2m_wlt - da_t2m_ctl).mean(dim="step")

# t2m is on the full grid_26_red cut-out, so mask outside the plot window like the precipitation maps above:
mask_window_t2m = ((np.rad2deg(grid_data["grid_26_red"]["clon"]) >= plot_window[0]) & (np.rad2deg(grid_data["grid_26_red"]["clon"]) <= plot_window[1]) &
                    (np.rad2deg(grid_data["grid_26_red"]["clat"]) >= plot_window[2]) & (np.rad2deg(grid_data["grid_26_red"]["clat"]) <= plot_window[3]))

In [ ]:
plot_wetdry_diff_maps(
    t2m_diff_wet, t2m_diff_dry,
    cbar_label=r"$\Delta$T$_\text{2m}$ in K",
    savepath="./figs/figure_s4.png",
    tri=grid_data["tri_26_red"], mask_window=mask_window_t2m.values,
)

## Lifting Codensation Level

In [ ]:
import metpy.calc as mpcalc
from metpy.units import units

dts_conv = pd.date_range("2021-07-13T09", "2021-07-13T18", freq="3h")
base_dir = "/automount/agh/s6tifohr/july21_eval/data"

# model level heights and land mask for the focus region (both time invariant)
_hhl = xr.open_dataset("invar/det_hhl_dom1.grb", engine="cfgrib",
                       backend_kwargs={"indexpath": ""})["HHL"].values[:, grid_data["focus_cells_26"]]
z_full = (0.5 * (_hhl[:-1] + _hhl[1:]))[::-1]   # full levels, surface first
z_sfc = _hhl[-1]
is_land = xr.open_dataset("data/fr_land.grb", engine="cfgrib",
                          backend_kwargs={"indexpath": ""})["lsm"].values[grid_data["focus_cells_26"]] > 0.5


def parcel_diags(run_dir, dt):
    """Mixed-layer parcel (lowest 50 hPa): its T, q and RH, and its LCL and LFC in m above ground."""
    folder = dt if dt.hour % 3 == 0 else dt.ceil("3h")   # each cycle folder holds dt-2 ... dt
    ds = (xr.open_dataset(f"{base_dir}/{run_dir}/{folder:%Y%m%d%H}/fc_R03B07_rea_ml.{dt:%Y%m%d%H}",
                          engine="cfgrib",
                          backend_kwargs={"indexpath": "",
                                          "filter_by_keys": {"typeOfLevel": "generalVerticalLayer"}})
          .rename({"values": "cell"}).isel(cell=grid_data["focus_cells_26_red"])[["t", "q", "pres"]]
          .load().isel(generalVerticalLayer=slice(None, None, -1)))   # surface first

    p = ds["pres"].values * units.Pa
    T = ds["t"].values * units.K
    Td = mpcalc.dewpoint_from_specific_humidity(p, ds["q"].values * units("kg/kg"))

    n = p.shape[1]
    out = {k: np.full(n, np.nan) for k in ("T", "q", "rh", "lcl", "lfc")}
    for c in range(n):
        par_p, par_T, par_Td = mpcalc.mixed_parcel(p[:, c], T[:, c], Td[:, c], depth=50 * units.hPa)
        prof = mpcalc.parcel_profile(p[:, c], par_T, par_Td)
        lcl_p, _ = mpcalc.lcl(par_p, par_T, par_Td)
        lfc_p, _ = mpcalc.lfc(p[:, c], T[:, c], Td[:, c], prof)

        out["T"][c] = par_T.to("K").m
        out["q"][c] = mpcalc.specific_humidity_from_dewpoint(par_p, par_Td).m * 1e3    # g/kg
        out["rh"][c] = mpcalc.relative_humidity_from_dewpoint(par_T, par_Td).m * 1e2   # %
        # model pressure -> height, interpolated in log p, referenced to the ground
        lnp = -np.log(p[:, c].m)
        out["lcl"][c] = np.interp(-np.log(lcl_p.to("Pa").m), lnp, z_full[:, c]) - z_sfc[c]
        if np.isfinite(lfc_p.m):
            out["lfc"][c] = np.interp(-np.log(lfc_p.to("Pa").m), lnp, z_full[:, c]) - z_sfc[c]
    return out


res_conv = {r: {k: [] for k in ("T", "q", "rh", "lcl", "lfc")} for r in ("BLK_CTL", "BLK_WLT")}
for dt in dts_conv:
    for run in res_conv:
        d = parcel_diags(run, dt)
        for k in d:
            res_conv[run][k].append(d[k])

C = {k: np.array(res_conv["BLK_CTL"][k])[:] for k in res_conv["BLK_CTL"]}
W = {k: np.array(res_conv["BLK_WLT"][k])[:] for k in res_conv["BLK_WLT"]}

In [ ]:
# Zoom window: 2 degrees wider than the focus region in every direction (the LCL diagnostic
# is only computed for focus-region cells, so the wide plot_window used for the E-P maps above
# would mostly show empty background).
plot_window_lcl = [FOCUS_REGION["x0"] - 2, FOCUS_REGION["x1"] + 2, FOCUS_REGION["y0"] - 2, FOCUS_REGION["y1"] + 2]

lon_deg, lat_deg = np.rad2deg(clon), np.rad2deg(clat)
lcl_ctl, lcl_dry = C["lcl"].mean(axis=0), W["lcl"].mean(axis=0)

vmin_raw = float(min(np.nanmin(lcl_ctl), np.nanmin(lcl_dry)))
vmax_raw = float(max(np.nanmax(lcl_ctl), np.nanmax(lcl_dry)))
step = nice_step(vmax_raw - vmin_raw)
vmin = np.floor(vmin_raw / step) * step
vmax = np.ceil(vmax_raw / step) * step
levels = np.round(np.arange(vmin, vmax + step / 2, step), 10)

fig, axs = plt.subplots(1, 2, figsize=(9, 4.2), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=2.5)

for ax, field, title, char in zip(axs, [lcl_ctl, lcl_dry], ["CTL", "DRY"], ["(a)", "(b)"]):
    im = ax.tricontourf(lon_deg, lat_deg, field, levels=levels, extend="both")
    ax.set(title=title)

    ax.set_extent(plot_window_lcl, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.07, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold",
        color="white", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground="black")])

# Position must be read AFTER set_extent: cartopy's aspect-locking can grow the geo-axes box
# beyond what tight_layout originally allotted, so measuring it beforehand places the colorbar
# too high and it ends up overlapping the maps. The colorbar's *top* edge (its y0 plus its
# height) must sit strictly below pos.y0, not just its y0 -- otherwise the box itself still
# pokes up into the maps regardless of how small the nominal gap looks.
pos = axs[0].get_position()
cbar_gap, cbar_height = 0.015, 0.03
cbar_ax = fig.add_axes([0.3, pos.y0 - cbar_gap - cbar_height, 0.4, cbar_height])
fig.colorbar(im, cax=cbar_ax, orientation="horizontal", label="LCL height in m", ticks=levels)

plt.savefig("./figs/figure_s5.png", dpi=300, bbox_inches='tight', format='png')
plt.show()